## Setup

In [1]:
from jaxcmr.memorysearch import FittingStrategy, BaseCMR, InstanceCMR, predict_trials, predict_trial, experience, start_retrieving
import pytest
import jax
import json
from jax import numpy as jnp, vmap, tree_util, random, lax, jit
from jax.nn import sigmoid
from jaxcmr.datasets import load_data, generate_trial_mask
from jaxcmr.helpers import latin_hypercube_sampling, scale_to_bounds, log_likelihood, get_item_count
from functools import partial
from evosax import Strategies, ParameterReshaper, EvoState
from jax_tqdm import scan_tqdm
import numpy as np

In [2]:
def scale_params(params: dict, bounds: dict[str, list[float]]):
    return jax.tree_util.tree_map(
        lambda x, bound: bound[0] + sigmoid(x) * (bound[1] - bound[0]), params, bounds
    )


def latin_hypercube_initialize(state, rng):
    x = scale_to_bounds(
        latin_hypercube_sampling(rng, state.archive.shape[1], state.archive.shape[0]),
        -5,
        5,
    )
    return x, state.replace(archive=x)

In [3]:
parameter_path = "D:/data/base_cmr_parameters.json"
data_path = "D:/data/{}.h5"

model_create_fn = BaseCMR
data_tag = "HealyKahana2014"
trial_query = "data['listtype'] == -1"
rng = jax.random.PRNGKey(0)

with open(parameter_path, "r") as f:
    parameters = json.load(f)

data = load_data(data_path.format(data_tag))
trial_mask = generate_trial_mask(data, trial_query)
subject_trials = jnp.array(data["recalls"][trial_mask].reshape(126, 28, 16))
subject_presentations = jnp.array(data["pres_itemnos"][trial_mask].reshape(126, 28, 16))

No GPU/TPU found, falling back to CPU. (Set TF_CPP_MIN_LOG_LEVEL=0 and rerun for more info.)


## Compiled Scan

In [4]:
# 1) parametrization
subject_index = 11
popsize = 100
s_name = "DE"
assert subject_index > 0

# init parameters: model_create_fn, dataset, trial_query, base parameters, subject division

# 2) configuration
strategy = Strategies[s_name](popsize=popsize, num_dims=len(parameters["free"]))
es_params = strategy.default_params

item_counts = vmap(get_item_count)(subject_presentations[subject_index-1])
max_item_count = max(item_counts).item()
reshape_params = ParameterReshaper(
    {key: parameters["fixed"][key] for key in parameters["free"]}
).reshape

# 3) function(s)
_predict_trials = vmap(predict_trials, in_axes=(None, None, None, 0))

def es_step(state_input, tmp):
    rng, state = state_input
    rng, rng_iter = random.split(rng)

    x, state = lax.cond(
        state.gen_counter == 0,
        lambda _: latin_hypercube_initialize(state, rng_iter),
        lambda _: strategy.ask(rng_iter, state, es_params),
        operand=None,
    )

    x_dict = scale_params(reshape_params(x), parameters["free"])
    fitness = _predict_trials(
        model_create_fn, max_item_count, subject_trials[subject_index-1], x_dict)
    state = strategy.tell(x, fitness, state, es_params)
    return [rng, state], fitness[jnp.argmin(fitness)]

# 4) scan

rng_init, rng = random.split(rng)
state = strategy.initialize(rng_init, es_params)

# final_state, scan_out = jax.lax.scan(es_step, [rng, state], [jnp.zeros(1000)])
final_state, scan_out = jax.lax.scan(es_step, [rng, state], None, length=1000)
print(jnp.min(scan_out))
scan_out

ParameterReshaper: 12 parameters detected for optimization.
642.7887


Array([1019.2505 ,  866.21234,  827.4328 ,  841.9862 ,  836.01514,
        831.4868 ,  866.7089 ,  867.36707,  831.7712 ,  874.942  ,
        822.8917 ,  703.4485 ,  739.057  ,  732.09454,  727.5183 ,
        731.6456 ,  716.6737 ,  729.44025,  734.6312 ,  707.6557 ,
        709.9344 ,  740.7058 ,  709.1741 ,  732.7277 ,  697.10443,
        708.6859 ,  703.4323 ,  713.1456 ,  705.8755 ,  684.3705 ,
        716.4987 ,  707.259  ,  687.13043,  679.6045 ,  707.35876,
        700.6834 ,  701.833  ,  696.0613 ,  686.1534 ,  701.0898 ,
        686.2342 ,  683.78064,  688.4157 ,  682.22705,  684.4439 ,
        687.0835 ,  680.03015,  680.77954,  687.9591 ,  681.9487 ,
        681.52   ,  679.63873,  681.96924,  679.05585,  675.2229 ,
        682.0049 ,  679.0381 ,  676.34814,  679.30505,  674.42957,
        675.8008 ,  675.8008 ,  679.49896,  675.8916 ,  671.3423 ,
        672.61426,  669.0938 ,  667.81085,  674.4507 ,  670.42035,
        670.376  ,  666.3731 ,  670.5933 ,  665.7969 ,  664.75

- Initialization step could be compiled along with the scan step.
- Instead of a set number of iterations, use `lax.while_loop` with a stopping condition based on the absolute change in the loss function value.
- Use `vmap` or `map` to fit multiple datasets at once.
- `init`, `ask`, and `tell` can all be mapped over a list of models or datasets.
- Mind how often compilation happens.
- Use general loss functions for the same model across subjects.

## Vmap Es-Step

We will want to vmap the loss_fn twice. Once to handle multiple datasets (sets of subject responses), another to handle multiple N=popsize parameter configurations per dataset. If we can do that across init, ask, and tell, we can fit multiple datasets at once.

In [31]:
# 1) parametrization
# subject_index = 1
popsize = 100
s_name = "DE"
assert subject_index > 0
iterations = 1000

# init parameters: model_create_fn, dataset, trial_query, base parameters, subject division

# 2) configuration
strategy = Strategies[s_name](popsize=popsize, num_dims=len(parameters["free"]))
es_params = strategy.default_params

item_counts = vmap(get_item_count)(subject_presentations[subject_index-1])
max_item_count = max(item_counts).item()
reshape_params = ParameterReshaper(
    {key: parameters["fixed"][key] for key in parameters["free"]}
).reshape

# 3) function(s)
_predict_trials = vmap(predict_trials, in_axes=(None, None, None, 0))
batch_init = jax.vmap(strategy.initialize, in_axes=(0, None)) # unique rng per init
# batch_ask = jax.vmap(strategy.ask, in_axes=(0, 0, None)) # unique rng, state per ask
# batch_tell = jax.vmap(strategy.tell, in_axes=(0, 0, 0, None)) # unique x, fitness, state per tell

def es_step(state_input, tmp):
    rng, state, subject_index = state_input
    rng, rng_iter = random.split(rng)

    x, state = lax.cond(
        state.gen_counter == 0,
        lambda _: latin_hypercube_initialize(state, rng_iter),
        lambda _: strategy.ask(rng_iter, state, es_params),
        operand=None,
    )

    x_dict = scale_params(reshape_params(x), parameters["free"])
    fitness = _predict_trials(
        model_create_fn, max_item_count, subject_trials[subject_index], x_dict)
    state = strategy.tell(x, fitness, state, es_params)
    return [rng, state, subject_index], fitness[jnp.argmin(fitness)]

# 4) scan

rng_init, rng = random.split(rng)
state = batch_init(random.split(rng_init, len(subject_trials)), es_params)

# final_state, scan_out = jax.lax.scan(es_step, [rng, state], [jnp.zeros(1000)])
final_state, scan_out = jax.lax.scan(
    vmap(es_step), 
    [random.split(rng, len(subject_trials)), state, jnp.arange(len(subject_trials))], 
    None, length=iterations)
print(np.nanmean(scan_out[-1]))
scan_out[-1]

ParameterReshaper: 12 parameters detected for optimization.
312.93546


Array([[785.874  , 874.8101 , 672.67487, ..., 594.70734, 791.8585 ,
        755.8148 ],
       [763.3872 , 822.4446 , 655.7909 , ..., 612.3331 , 798.6415 ,
        731.70013],
       [750.7408 , 767.97076, 653.237  , ..., 615.13245, 813.1056 ,
        726.0422 ],
       ...,
       [689.40466, 503.56976, 485.19635, ..., 509.14142, 625.6073 ,
        651.3978 ],
       [689.40466, 503.56976, 485.19635, ..., 509.14133, 625.6073 ,
        651.3978 ],
       [689.40466, 503.56976, 485.19635, ..., 509.14142, 625.6073 ,
        651.3978 ]], dtype=float32)

In [35]:
param_path = "D:/data/results/{}_{}_{}.jsonl"
model_name = "Base_CMR"
peers_data_tag = "HealyKahana2014"

ignore_first_recall = False
with open(param_path.format(model_name, peers_data_tag, ignore_first_recall)) as f:
    param_list = [json.loads(line) for line in f.readlines()]

np.mean([each['likelihood'] for each in param_list])

588.6674000573537

## Local Vmap

In [35]:
# # 1) setup

# popsize = 15
# s_name = "DE"

# strategy = Strategies[s_name](popsize=popsize, num_dims=len(parameters["free"]))
# es_params = strategy.default_params

# # 2) configure vmap over rng and state
# batch_init = jax.vmap(strategy.initialize, in_axes=(0, None)) 
# batch_ask = jax.vmap(strategy.ask, in_axes=(0, 0, None))
# batch_tell = jax.vmap(strategy.tell, in_axes=(0, 0, 0, None))

# batch_latin_hypercube_initialize = jax.vmap(latin_hypercube_initialize, in_axes=(0, 0))

# # Vmap-composed objective function
# # first by parameter, then by subset of trials
# batch_predict = vmap(
#     vmap(predict_trials, in_axes=(None, None, None, 0)), in_axes=(None, None, 0, None)
# )


# # 3) configure scan
# def es_step(state_input, tmp):
#     rng, state = state_input
#     rng, rng_iter = random.split(rng)

#     x, state = lax.cond(
#         state.gen_counter == 0,
#         lambda _: batch_latin_hypercube_initialize(state, rng_iter),
#         lambda _: batch_ask(rng_iter, state, es_params),
#         operand=None,
#     )

#     x_dict = scale_params(reshape_params(x), parameters["free"])
#     fitness = loss_fn(x_dict)
#     state = strategy.tell(x, fitness, state, es_params)
#     return [rng, state], fitness[jnp.argmin(fitness)]


# # 4) run
# rng, rng_iter = random.split(rng)
# state = strategy.initialize(rng, es_params)

# # final_state, scan_out = jax.lax.scan(es_step, [rng, state], [jnp.zeros(1000)])
# final_state, scan_out = jax.lax.scan(es_step, [rng, state], None, length=1000)
# jnp.min(scan_out)

Array(642.77075, dtype=float32)

## Multiple Subjects: Map Function Approach
This approach allows easier code reuse since I don't need a special function for handling multiple subjects. Let's see what it would take to make this work. I will probably want to map over the scan like I current do for `predict_trial`.

That means one function for a single pass. This was `predict_and_simulate_retrieval` in the previous case. It's probably `es_step` in this case. Hopefully I can find a better name. Maybe `increment_fit`? It will return a new state and a fitness value.

Then another function that scans over `es_steps`. This was `predict_trial` in the previous case. It's probably `fit_to_trials` in this case. Hopefully I can find a better name. It will return a final state and a list of states (probably best fitness at each step). I may move this to a while loop instead of scan.

A wrapper function that maps over subjects. This was `predict_trials` mapping over trial vectors in the previous case. Call it `fit_to_subjects`. This one will map over sets of trial vectors. After post-processing the output, it will return a list of fitting parameter configurations as dictionaries.

Main obstacle is that I need to use the same loss function across calls to `fit_to_trials`. This means `create_predict_fn` produces functions that are too specialized. I will ultimately need to address the item_count issue if I want a clean implementation. 

What would that mean? Basically, set array sizes across the model by a max_item_count parameter. But dynamically configure by an item_count parameter, ignoring additional slots when item_count < max_item_count. This would require a rewrite of most of the model functions. Well, maybe not -- perhaps just the accessor functions. No, if the size of arrays returned by accessors depends on item_count, that would raise an error. Ugh. Let's do it; I hate worrying about this.